# 4. DSSM Candidates

Ноутбук строит DSSM-кандидатов по заранее обученному checkpoint и объединяет их с базовым retrieval candidate pool.

Этот ноутбук не обучает DSSM заново. Он выполняет:

- загрузку checkpoint `dl_dssm_final/val/model.pt`;
- генерацию DSSM-кандидатов для `ranker_train`, `val` и `test`;
- ANN retrieval через HNSW/FAISS/torch fallback;
- merge с base retrieval;
- расчет validation-метрик.

In [1]:

from __future__ import annotations

import gc
import json
import math
import shutil
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm, trange

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import hnswlib
    HAS_HNSWLIB = True
except Exception:
    HAS_HNSWLIB = False

try:
    import faiss
    HAS_FAISS = True
except Exception:
    HAS_FAISS = False


def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "datamart").exists() or (p / "data" / "raw").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с data/datamart или data/raw")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = DATA_DIR / "processed" / "dl_candidate_generators_dssm_final"
MODELS_DIR = PROJECT_ROOT / "models" / "dl_candidate_generators_dssm_final"
SOURCE_NAME = "dl_dssm_final"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


@dataclass(frozen=True)
class DLInferenceMergeConfig:
    # Dates are aligned with candidate/reranker notebooks.
    ranker_train_context_start: date = date(2025, 4, 18)
    ranker_train_context_end: date = date(2025, 6, 3)
    ranker_train_target_start: date = date(2025, 6, 4)
    ranker_train_target_end: date = date(2025, 6, 17)

    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    n_shards: int = 128
    source_name: str = SOURCE_NAME
    output_merged_name: str = "merged_with_dl"

    # Use the already trained validation checkpoint. Do not retrain DSSM here.
    checkpoint_split: str = "val"
    checkpoint_path: str = ""  # empty -> models/.../dl_dssm_final/val/model.pt

    # Build only missing DL sources by default. val exists already; ranker_train/test are usually missing.
    splits_to_build_dl: tuple[str, ...] = ("ranker_train", "test")
    splits_to_merge: tuple[str, ...] = ("ranker_train", "val", "test")

    # Rebuild controls.
    rebuild_dl_sources: bool = False       # False = do not overwrite existing val source
    rebuild_missing_dl_sources: bool = True
    rebuild_ann_index: bool = False        # load existing val HNSW if present
    rebuild_merged_with_dl: bool = True

    # DSSM source generation.
    sequence_max_len: int = 80
    dl_candidates_per_user: int = 300
    ann_oversample: int = 2000
    retrieval_user_batch_size: int = 4096
    retrieval_flush_rows: int = 2_000_000
    hnsw_ef_search: int = 500
    hnsw_m: int = 24
    hnsw_ef_construction: int = 120
    hnsw_num_threads: int = -1
    hnsw_add_batch_size: int = 50_000
    filter_seen_context_items: bool = False

    # Safe merge policy.
    base_keep_per_user: int = 700
    dl_keep_per_user: int = 300
    total_keep_per_user: int = 1000  # base 700 + up to all 300 DSSM candidates

    # Calibration used in final DSSM notebook.
    dssm_candidate_pop_alpha: float = 0.04
    dssm_candidate_cat_l0_alpha: float = 0.08
    dssm_candidate_catalog_alpha: float = 0.03
    dssm_candidate_rank_alpha: float = 0.0005

    # Model architecture. These are also checked against checkpoint config when available.
    dssm_embedding_dim: int = 256
    dssm_item_id_dim: int = 256
    dssm_catalog_dim: int = 64
    dssm_cat_l0_dim: int = 32
    dssm_user_id_dim: int = 64
    dssm_hidden_dims: tuple[int, ...] = (1024, 512, 256)
    dssm_dropout: float = 0.12
    dssm_temperature_init: float = 0.07
    dssm_use_user_id_embedding: bool = False
    dssm_use_fclip_projection: bool = True
    fclip_project_dim: int = 64

    # Interaction sequence construction.
    weight_preset: str = "hybrid"
    event_weight_page_view: float = 1.0
    event_weight_view_description: float = 2.0
    event_weight_review_view: float = 3.0
    event_weight_favorite: float = 10.0
    event_weight_to_cart: float = 30.0
    order_weight: float = 60.0
    weak_max_cap: float = 10.0
    min_item_weight: float = 0.1
    time_decay_enabled: bool = True
    time_decay_tau_days: float = 30.0
    time_decay_min_factor: float = 0.05

    metric_ks: tuple[int, ...] = (20, 50, 100, 200, 300, 700, 900, 1000)
    evaluate_val: bool = True
    run_unique_positive_report: bool = True
    run_quota_ablation: bool = True
    quota_values: tuple[int, ...] = (50, 100, 200, 300)


CFG = DLInferenceMergeConfig()


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def cfg_json() -> str:
    return json.dumps({k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()}, ensure_ascii=False, indent=2)


print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("HAS_HNSWLIB:", HAS_HNSWLIB, "HAS_FAISS:", HAS_FAISS)
print("Config:")
print(cfg_json())


PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
DEVICE: cuda
HAS_HNSWLIB: True HAS_FAISS: True
Config:
{
  "ranker_train_context_start": "2025-04-18",
  "ranker_train_context_end": "2025-06-03",
  "ranker_train_target_start": "2025-06-04",
  "ranker_train_target_end": "2025-06-17",
  "val_context_start": "2025-04-18",
  "val_context_end": "2025-06-17",
  "val_target_start": "2025-06-18",
  "val_target_end": "2025-07-01",
  "full_context_start": "2025-05-02",
  "full_context_end": "2025-07-01",
  "test_target_start": "2025-07-02",
  "test_target_end": "2025-07-15",
  "n_shards": 128,
  "source_name": "dl_dssm_final",
  "output_merged_name": "merged_with_dl",
  "checkpoint_split": "val",
  "checkpoint_path": "",
  "splits_to_build_dl": [
    "ranker_train",
    "test"
  ],
  "splits_to_merge": [
    "ranker_train",
    "val",
    "test"
  ],
  "rebuild_dl_sources": false,
  "rebuild_missing_dl_sources": true,
  "rebuild_ann_index": false,
  "rebuild_merged_with_dl": true,
  "sequence_ma

In [2]:

# -----------------------------
# I/O helpers
# -----------------------------

def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def candidate_files(path: Path) -> list[Path]:
    return sorted(path.rglob("*.parquet")) if path.exists() else []


def candidate_dir_looks_complete(path: Path, *, min_shards: int | None = None) -> bool:
    files = candidate_files(path)
    if not files:
        return False
    if min_shards is None:
        return True
    shard_dirs = {p.parent.name for p in files if p.parent.name.startswith("user_shard=") and p.stat().st_size > 0}
    return len(shard_dirs) >= min_shards


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def scan_candidate_shard_or_none(base_dir: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = base_dir / f"user_shard={shard_id}"
    if not path_has_parquet(shard_dir):
        return None
    return pl.scan_parquet(str(shard_dir / "*.parquet"), hive_partitioning=True)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def clean_output_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def resolve_manifest_candidate_dir(split: str) -> Path | None:
    manifest_path = CANDIDATES_DIR / "manifest_candidate_generators_enhanced.json"
    if not manifest_path.exists():
        return None
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: failed to read manifest {rel(manifest_path)}: {e}")
        return None
    key_by_split = {
        "ranker_train": "ranker_train_candidates",
        "val": "val_candidates",
        "test": "test_candidates",
    }
    raw = manifest.get("outputs", {}).get(key_by_split.get(split, ""))
    if not raw:
        return None
    p = Path(raw)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return p if candidate_dir_looks_complete(p) else None


def resolve_base_candidate_dir(split: str) -> Path | None:
    p = resolve_manifest_candidate_dir(split)
    if p is not None:
        return p
    for p in [
        CANDIDATES_DIR / "merged" / split,
        CANDIDATES_DIR / "merged_enhanced" / split,
        CANDIDATES_DIR / "enhanced_merged" / split,
    ]:
        if candidate_dir_looks_complete(p):
            return p
    return None


def resolve_dl_candidate_dir(split: str) -> Path | None:
    p = CANDIDATES_DIR / CFG.source_name / split
    return p if candidate_dir_looks_complete(p) else None


def get_checkpoint_path() -> Path:
    if CFG.checkpoint_path:
        p = Path(CFG.checkpoint_path)
        return p if p.is_absolute() else PROJECT_ROOT / p
    return MODELS_DIR / CFG.source_name / CFG.checkpoint_split / "model.pt"


def read_base_users(split: str) -> np.ndarray:
    cache = PROCESSED_DIR / "inference_from_checkpoint" / "base_users"
    ensure_dir(cache)
    out = cache / f"base_users_{split}.npy"
    if out.exists():
        arr = np.load(out)
        print(f"{split}: base users loaded -> {rel(out)}, users={len(arr):,}")
        return arr.astype(np.int64)

    base_dir = resolve_base_candidate_dir(split)
    if base_dir is None:
        raise FileNotFoundError(f"Base candidates not found for split={split}")

    users = []
    for shard_id in trange(CFG.n_shards, desc=f"Read base users {split}"):
        lf = scan_candidate_shard_or_none(base_dir, shard_id)
        if lf is None:
            continue
        part = collect_streaming(lf.select(pl.col("user_id").cast(pl.Int64).unique()))
        users.append(part["user_id"].to_numpy())
    if not users:
        raise RuntimeError(f"No base users read for split={split}")
    arr = np.unique(np.concatenate(users).astype(np.int64))
    np.save(out, arr)
    print(f"{split}: base users saved -> {rel(out)}, users={len(arr):,}")
    return arr


In [3]:

# -----------------------------
# Context sequences with checkpoint item vocabulary
# -----------------------------

def events_path(level: str) -> Path:
    return AGG_DIR / "events" / level


def orders_path(level: str) -> Path:
    return AGG_DIR / "orders" / level


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(events_path(level), hive_partitioning=True)
    return lf.filter((pl.col("event_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("event_date").cast(pl.Date) <= pl.lit(end)))


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(orders_path(level), hive_partitioning=True)
    return lf.filter((pl.col("created_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("created_date").cast(pl.Date) <= pl.lit(end)))


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Sequence[str]) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    exprs = []
    for c in cols:
        if c not in existing:
            exprs.append(pl.lit(0, dtype=pl.Float32).alias(c))
        else:
            exprs.append(pl.col(c).fill_null(0).cast(pl.Float32).alias(c))
    return lf.with_columns(exprs)


def split_spec(split_name: str) -> tuple[date, date, date, date]:
    if split_name == "ranker_train":
        return CFG.ranker_train_context_start, CFG.ranker_train_context_end, CFG.ranker_train_target_start, CFG.ranker_train_target_end
    if split_name == "val":
        return CFG.val_context_start, CFG.val_context_end, CFG.val_target_start, CFG.val_target_end
    if split_name == "test":
        return CFG.full_context_start, CFG.full_context_end, CFG.test_target_start, CFG.test_target_end
    raise ValueError(split_name)


def weight_values() -> dict[str, float]:
    if CFG.weight_preset == "ecup_soft":
        return {"page_view": 2.0, "view_description": 3.0, "review_view": 3.0, "favorite": 5.0, "to_cart": 1.0, "order": 5.0}
    if CFG.weight_preset == "hybrid":
        return {"page_view": 1.0, "view_description": 2.0, "review_view": 3.0, "favorite": 10.0, "to_cart": 30.0, "order": 60.0}
    return {
        "page_view": CFG.event_weight_page_view,
        "view_description": CFG.event_weight_view_description,
        "review_view": CFG.event_weight_review_view,
        "favorite": CFG.event_weight_favorite,
        "to_cart": CFG.event_weight_to_cart,
        "order": CFG.order_weight,
    }


def time_decay_expr(date_col: str, context_end: date) -> pl.Expr:
    if not CFG.time_decay_enabled:
        return pl.lit(1.0, dtype=pl.Float32)
    age_days = (pl.lit(context_end) - pl.col(date_col).cast(pl.Date)).dt.total_days().cast(pl.Float32)
    return (-age_days / float(CFG.time_decay_tau_days)).exp().clip(lower_bound=CFG.time_decay_min_factor, upper_bound=1.0).cast(pl.Float32)


def build_interactions_lf(start: date, end: date, base_users: np.ndarray | None = None) -> pl.LazyFrame:
    w = weight_values()
    user_filter = None
    if base_users is not None:
        users_df = pl.DataFrame({"user_id": base_users.astype(np.int64)}).lazy()
        user_filter = users_df

    event_cols = ["cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
    e = ensure_numeric_cols(read_events_lf("user_item", start, end), event_cols)
    if user_filter is not None:
        e = e.join(user_filter, on="user_id", how="inner")
    event_weight = (
        pl.col("cnt_page_view").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["page_view"]
        + pl.col("cnt_view_description").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["view_description"]
        + pl.col("cnt_review_view").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["review_view"]
        + (pl.col("cnt_favorite").fill_null(0) > 0).cast(pl.Float32) * w["favorite"]
        + (pl.col("cnt_to_cart").fill_null(0) > 0).cast(pl.Float32) * w["to_cart"]
    ) * time_decay_expr("event_date", end)
    e = (
        e.with_columns(event_weight.alias("weight"), pl.col("event_date").cast(pl.Date).alias("dt"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32))
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    o = ensure_numeric_cols(read_orders_lf("user_item", start, end), ["cnt_orders"])
    if user_filter is not None:
        o = o.join(user_filter, on="user_id", how="inner")
    order_weight = (pl.col("cnt_orders").fill_null(0).cast(pl.Float32) * w["order"] * time_decay_expr("created_date", end)).cast(pl.Float32)
    o = (
        o.with_columns(order_weight.alias("weight"), pl.col("created_date").cast(pl.Date).alias("dt"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32))
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    return (
        pl.concat([e, o], how="diagonal_relaxed")
        .group_by(["user_id", "item_id", "dt"])
        .agg(pl.sum("weight").cast(pl.Float32).alias("weight"))
        .filter(pl.col("weight") > 0)
    )


def build_sequences_for_split_with_checkpoint_vocab(split_name: str, item2idx: dict[int, int]) -> tuple[np.ndarray, dict[int, np.ndarray], dict[int, set[int]]]:
    cache = PROCESSED_DIR / "inference_from_checkpoint" / "sequences" / split_name
    ensure_dir(cache)
    user_ids_path = cache / "user_ids.npy"
    seqs_path = cache / "seqs.npy"
    meta_path = cache / "meta.json"

    if user_ids_path.exists() and seqs_path.exists() and meta_path.exists():
        user_ids = np.load(user_ids_path).astype(np.int64)
        seqs_obj = np.load(seqs_path, allow_pickle=True)
        seqs = [np.asarray(x, dtype=np.int32) for x in seqs_obj.tolist()]
        user_seq = {int(u): seqs[i] for i, u in enumerate(user_ids)}
        seen_by_user = {int(u): set() for u in user_ids}  # not needed unless filtering seen items
        print(f"{split_name}: checkpoint-vocab sequences reused -> {rel(cache)}, users={len(user_ids):,}")
        return user_ids, user_seq, seen_by_user

    base_users = read_base_users(split_name)
    start, end, _, _ = split_spec(split_name)
    item_map = pl.DataFrame({
        "item_id": np.fromiter(item2idx.keys(), dtype=np.int64),
        "item_idx": np.fromiter(item2idx.values(), dtype=np.int32),
    }).lazy()

    print(f"{split_name}: build context sequences with checkpoint item vocab, context={start}..{end}, base_users={len(base_users):,}")
    lf = (
        build_interactions_lf(start, end, base_users=base_users)
        .join(item_map, on="item_id", how="inner")
        .select("user_id", "item_idx", "dt", "weight")
        .sort(["user_id", "dt", "weight", "item_idx"], descending=[False, False, False, False])
        .group_by("user_id", maintain_order=True)
        .agg(pl.col("item_idx").tail(CFG.sequence_max_len).alias("seq"))
        .sort("user_id")
    )
    seq_df = collect_streaming(lf)
    if seq_df.is_empty():
        raise RuntimeError(f"No sequences built for split={split_name}")

    user_ids = seq_df["user_id"].to_numpy().astype(np.int64)
    seqs = [np.asarray(x, dtype=np.int32) for x in seq_df["seq"].to_list()]
    np.save(user_ids_path, user_ids)
    np.save(seqs_path, np.asarray(seqs, dtype=object), allow_pickle=True)
    meta_path.write_text(json.dumps({
        "split": split_name,
        "context_start": str(start),
        "context_end": str(end),
        "base_users": int(len(base_users)),
        "users_with_checkpoint_vocab_history": int(len(user_ids)),
        "sequence_max_len": int(CFG.sequence_max_len),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{split_name}: checkpoint-vocab sequences saved -> {rel(cache)}, users={len(user_ids):,}")
    return user_ids, {int(u): seqs[i] for i, u in enumerate(user_ids)}, {int(u): set() for u in user_ids}


In [4]:

# -----------------------------
# DSSM model definition and checkpoint loading
# -----------------------------

class MLP(nn.Module):
    def __init__(self, dims: Sequence[int], dropout: float, final_norm: bool = False):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.GELU())
                layers.append(nn.LayerNorm(dims[i + 1]))
                layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
        self.final_norm = final_norm

    def forward(self, x):
        x = self.net(x)
        return F.normalize(x, dim=-1) if self.final_norm else x


class AttentionPooling(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        scores = self.scorer(x).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e4)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)


class DSSMMaxTwoTower(nn.Module):
    def __init__(
        self,
        num_items: int,
        num_users: int,
        catalog_idx_by_item_idx: np.ndarray,
        cat_l0_idx_by_item_idx: np.ndarray,
        num_catalogs: int,
        num_cat_l0: int,
        *,
        fclip_reduced: np.ndarray | None = None,
        output_dim: int = CFG.dssm_embedding_dim,
        dropout: float = CFG.dssm_dropout,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.output_dim = int(output_dim)
        self.use_user_id = bool(CFG.dssm_use_user_id_embedding)

        self.item_id_embedding = nn.Embedding(num_items, CFG.dssm_item_id_dim, padding_idx=0)
        self.catalog_embedding = nn.Embedding(max(1, num_catalogs), CFG.dssm_catalog_dim, padding_idx=0)
        self.cat_l0_embedding = nn.Embedding(max(1, num_cat_l0), CFG.dssm_cat_l0_dim, padding_idx=0)

        self.register_buffer("catalog_idx_by_item_idx", torch.as_tensor(catalog_idx_by_item_idx, dtype=torch.long), persistent=False)
        self.register_buffer("cat_l0_idx_by_item_idx", torch.as_tensor(cat_l0_idx_by_item_idx, dtype=torch.long), persistent=False)

        clip_dim = 0
        if fclip_reduced is not None:
            clip_arr = np.asarray(fclip_reduced, dtype=np.float32)
            clip_dim = int(clip_arr.shape[1])
            self.register_buffer("fclip_table", torch.from_numpy(clip_arr), persistent=False)
        else:
            self.register_buffer("fclip_table", torch.empty(0, 0, dtype=torch.float32), persistent=False)

        item_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim + clip_dim
        self.item_base = nn.Sequential(
            nn.Linear(item_input_dim, CFG.dssm_hidden_dims[0]),
            nn.GELU(),
            nn.LayerNorm(CFG.dssm_hidden_dims[0]),
            nn.Dropout(dropout),
            nn.Linear(CFG.dssm_hidden_dims[0], output_dim),
            nn.LayerNorm(output_dim),
        )
        self.item_tower = MLP([output_dim, *CFG.dssm_hidden_dims[1:], output_dim], dropout, final_norm=True)

        history_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim
        self.history_project = nn.Sequential(
            nn.Linear(history_input_dim, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
        )

        user_input_dim = output_dim * 3
        if self.use_user_id:
            self.user_embedding = nn.Embedding(num_users, CFG.dssm_user_id_dim)
            user_input_dim += CFG.dssm_user_id_dim
        else:
            self.user_embedding = None

        self.attention_pool = AttentionPooling(output_dim, max(64, output_dim // 2), dropout)
        self.user_tower = MLP([user_input_dim, *CFG.dssm_hidden_dims, output_dim], dropout, final_norm=True)
        self.temperature = nn.Parameter(torch.tensor(float(CFG.dssm_temperature_init)))

    def _item_base_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        catalog = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        parts = [
            self.item_id_embedding(item_idx),
            self.catalog_embedding(catalog.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ]
        if self.fclip_table.numel() > 0:
            parts.append(self.fclip_table[item_idx].to(parts[0].device))
        return torch.cat(parts, dim=-1)

    def _history_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        catalog = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        return torch.cat([
            self.item_id_embedding(item_idx),
            self.catalog_embedding(catalog.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ], dim=-1)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        base = self.item_base(self._item_base_features(item_idx))
        return self.item_tower(base)

    def encode_user(self, user_idx: torch.Tensor, hist_item_idx: torch.Tensor) -> torch.Tensor:
        mask = hist_item_idx > 0
        tok = self.history_project(self._history_features(hist_item_idx))
        mask_f = mask.unsqueeze(-1).float()
        mean = (tok * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        att = self.attention_pool(tok, mask)
        # Sequences are right-aligned, so the last real event is at -1 for users with non-empty history.
        last = tok[:, -1, :]
        parts = [mean, att, last]
        if self.use_user_id and self.user_embedding is not None:
            parts.append(self.user_embedding(user_idx.clamp(0, self.user_embedding.num_embeddings - 1)))
        return self.user_tower(torch.cat(parts, dim=-1))


def load_side_arrays_for_checkpoint(item_id_by_idx: np.ndarray) -> dict:
    side_dir = PROCESSED_DIR / "item_side"
    side_path = side_dir / f"item_side_{CFG.checkpoint_split}.npz"
    meta_path = side_dir / f"item_side_{CFG.checkpoint_split}_meta.json"
    if not side_path.exists() or not meta_path.exists():
        raise FileNotFoundError(
            f"Cached side arrays are required for inference-only run: {rel(side_path)} / {rel(meta_path)}. "
            "Do not rebuild DSSM; first restore these files or rerun only item_side preparation."
        )
    data = np.load(side_path)
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    catalog_idx = data["catalog_idx_by_item_idx"].astype(np.int64)
    cat_l0_idx = data["cat_l0_idx_by_item_idx"].astype(np.int64)
    if len(catalog_idx) != len(item_id_by_idx) or len(cat_l0_idx) != len(item_id_by_idx):
        raise ValueError(f"Side array length mismatch: side={len(catalog_idx)}, checkpoint items={len(item_id_by_idx)}")

    fclip = None
    fclip_path = side_dir / f"fclip_reduced_{CFG.checkpoint_split}_dim={CFG.fclip_project_dim}.npy"
    if CFG.dssm_use_fclip_projection:
        if not fclip_path.exists():
            raise FileNotFoundError(f"fclip reduced matrix not found: {rel(fclip_path)}")
        fclip = np.load(fclip_path, mmap_mode="r")
        if len(fclip) != len(item_id_by_idx):
            raise ValueError(f"fclip length mismatch: fclip={len(fclip)}, checkpoint items={len(item_id_by_idx)}")

    return {
        "catalog_idx_by_item_idx": catalog_idx,
        "cat_l0_idx_by_item_idx": cat_l0_idx,
        "num_catalogs": int(meta["num_catalogs"]),
        "num_cat_l0": int(meta["num_cat_l0"]),
        "fclip_reduced": fclip,
    }


def load_dssm_checkpoint() -> tuple[nn.Module, np.ndarray, dict[int, int]]:
    ckpt_path = get_checkpoint_path()
    if not ckpt_path.exists():
        raise FileNotFoundError(f"DSSM checkpoint not found: {rel(ckpt_path)}")
    print(f"Load checkpoint: {rel(ckpt_path)}")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    item_id_by_idx = np.asarray(ckpt["item_id_by_idx"], dtype=np.int64)
    item2idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
    side = load_side_arrays_for_checkpoint(item_id_by_idx)
    model = DSSMMaxTwoTower(
        num_items=len(item_id_by_idx),
        num_users=1,  # user_id embedding is disabled in final DSSM
        catalog_idx_by_item_idx=side["catalog_idx_by_item_idx"],
        cat_l0_idx_by_item_idx=side["cat_l0_idx_by_item_idx"],
        num_catalogs=side["num_catalogs"],
        num_cat_l0=side["num_cat_l0"],
        fclip_reduced=side["fclip_reduced"],
        output_dim=CFG.dssm_embedding_dim,
        dropout=CFG.dssm_dropout,
    )
    state = ckpt["model_state_dict"]
    state = {k: v for k, v in state.items() if k != "fclip_embedding.weight"}
    res = model.load_state_dict(state, strict=False)
    missing = [k for k in res.missing_keys if not k.startswith("fclip_table")]
    unexpected = [k for k in res.unexpected_keys if k != "fclip_embedding.weight"]
    if missing or unexpected:
        print("load_state_dict missing:", missing[:10], "unexpected:", unexpected[:10])
    model.eval().to(DEVICE)
    print(f"Checkpoint loaded: items={len(item_id_by_idx)-1:,}, source={SOURCE_NAME}")
    return model, item_id_by_idx, item2idx


In [5]:

# -----------------------------
# ANN retrieval and DL candidate generation
# -----------------------------

@torch.no_grad()
def compute_item_vectors(model: nn.Module, num_items: int, batch_size: int = 65536) -> np.ndarray:
    model.eval()
    vecs = []
    for start in trange(1, num_items, batch_size, desc="Encode items"):
        end = min(num_items, start + batch_size)
        idx = torch.arange(start, end, device=DEVICE, dtype=torch.long)
        v = model.encode_item_idx(idx).detach().float().cpu().numpy()
        vecs.append(v.astype(np.float32))
    return np.vstack(vecs).astype(np.float32)


@torch.no_grad()
def encode_user_batch(model: nn.Module, hists: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    model.eval()
    out = []
    for start in range(0, len(hists), batch_size):
        end = min(len(hists), start + batch_size)
        uidx = torch.zeros(end - start, device=DEVICE, dtype=torch.long)
        hist = torch.from_numpy(hists[start:end].astype(np.int64)).to(DEVICE)
        v = model.encode_user(uidx, hist).detach().float().cpu().numpy()
        out.append(v.astype(np.float32))
    return np.vstack(out).astype(np.float32) if out else np.empty((0, CFG.dssm_embedding_dim), dtype=np.float32)


@dataclass
class ANNIndex:
    backend: str
    index: object | None
    n_items: int
    item_vecs: np.ndarray | None = None

    def query(self, user_vecs: np.ndarray, topk: int) -> tuple[np.ndarray, np.ndarray]:
        topk = int(min(topk, self.n_items))
        user_vecs = np.asarray(user_vecs, dtype=np.float32)
        if self.backend == "hnswlib":
            labels, dist = self.index.knn_query(user_vecs, k=topk)
            return labels.astype(np.int64), (1.0 - dist).astype(np.float32)
        if self.backend == "faiss":
            scores, labels = self.index.search(user_vecs, topk)
            return labels.astype(np.int64), scores.astype(np.float32)
        if self.backend == "torch":
            item_t = torch.from_numpy(self.item_vecs).to(DEVICE).float()
            item_t = F.normalize(item_t, dim=1)
            all_idx, all_scores = [], []
            bs = 512
            for s in range(0, len(user_vecs), bs):
                u = torch.from_numpy(user_vecs[s:s+bs]).to(DEVICE).float()
                u = F.normalize(u, dim=1)
                scores = u @ item_t.T
                vals, idx = torch.topk(scores, k=topk, dim=1)
                all_idx.append(idx.detach().cpu().numpy().astype(np.int64))
                all_scores.append(vals.detach().cpu().numpy().astype(np.float32))
            return np.vstack(all_idx), np.vstack(all_scores)
        raise ValueError(self.backend)


def load_or_build_ann_index(model: nn.Module, item_id_by_idx: np.ndarray) -> ANNIndex:
    n_items = len(item_id_by_idx) - 1
    ann_dir = MODELS_DIR / CFG.source_name / CFG.checkpoint_split
    hnsw_path = ann_dir / f"{CFG.source_name}_{CFG.checkpoint_split}_hnsw.bin"
    if HAS_HNSWLIB and hnsw_path.exists() and not CFG.rebuild_ann_index:
        idx = hnswlib.Index(space="cosine", dim=CFG.dssm_embedding_dim)
        idx.load_index(str(hnsw_path), max_elements=n_items)
        idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
        print(f"HNSW loaded: {rel(hnsw_path)}")
        return ANNIndex("hnswlib", idx, n_items)

    item_vecs = compute_item_vectors(model, len(item_id_by_idx))
    if HAS_HNSWLIB:
        ensure_dir(ann_dir)
        idx = hnswlib.Index(space="cosine", dim=item_vecs.shape[1])
        idx.init_index(max_elements=item_vecs.shape[0], ef_construction=CFG.hnsw_ef_construction, M=CFG.hnsw_m)
        for s in trange(0, item_vecs.shape[0], CFG.hnsw_add_batch_size, desc="Build HNSW"):
            e = min(item_vecs.shape[0], s + CFG.hnsw_add_batch_size)
            idx.add_items(item_vecs[s:e], np.arange(s, e, dtype=np.int64), num_threads=CFG.hnsw_num_threads)
        idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
        idx.save_index(str(hnsw_path))
        print(f"HNSW built: {rel(hnsw_path)}")
        return ANNIndex("hnswlib", idx, item_vecs.shape[0])

    print("hnswlib is unavailable; using torch exact retrieval. This can be slow.")
    return ANNIndex("torch", None, item_vecs.shape[0], item_vecs=item_vecs)


def _safe_model_buffer_numpy(model: nn.Module, name: str, length: int) -> np.ndarray:
    arr = getattr(model, name, None)
    if torch.is_tensor(arr):
        out = arr.detach().cpu().numpy()
        if len(out) == length:
            return out.astype(np.int32)
    return np.zeros(length, dtype=np.int32)


def _history_pref_maps(hist_idx: np.ndarray, value_by_item_idx: np.ndarray) -> dict[int, float]:
    hist_idx = hist_idx[hist_idx > 0]
    if len(hist_idx) == 0:
        return {}
    vals = value_by_item_idx[np.clip(hist_idx, 0, len(value_by_item_idx) - 1)]
    vals = vals[vals > 0]
    if len(vals) == 0:
        return {}
    unique, cnt = np.unique(vals.astype(np.int32), return_counts=True)
    denom = float(cnt.sum())
    return {int(k): float(v) / denom for k, v in zip(unique, cnt)}


def load_item_pop_prior_for_checkpoint_vocab(split_name: str, item_id_by_idx: np.ndarray) -> np.ndarray:
    # Try checkpoint-specific sequence cache first; otherwise compute a cheap zero prior.
    path = PROCESSED_DIR / "sequences" / CFG.checkpoint_split / "item_weight_by_idx.npy"
    if path.exists():
        arr = np.load(path).astype(np.float32)
        pop = np.log1p(arr) if len(arr) == len(item_id_by_idx) else np.zeros(len(item_id_by_idx), dtype=np.float32)
    else:
        pop = np.zeros(len(item_id_by_idx), dtype=np.float32)
    if np.nanmax(pop) > 0:
        q = np.nanquantile(pop[1:], 0.995) if len(pop) > 1 else np.nanmax(pop)
        q = float(q) if np.isfinite(q) and q > 0 else float(np.nanmax(pop))
        pop = np.clip(pop / max(q, 1e-6), 0.0, 1.0)
    pop[0] = 0.0
    return pop.astype(np.float32)


def write_dl_buffer_to_shards(rows: list[tuple], out_dir: Path, source_name: str, part_id: int) -> int:
    if not rows:
        return 0
    df = pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "score", "rank", "raw_dssm_score", "pop_prior", "cat_l0_pref", "catalog_pref"],
    )
    df["source"] = source_name
    df[f"{source_name}_score"] = df["score"].astype("float32")
    df[f"{source_name}_rank"] = df["rank"].astype("int32")
    df[f"{source_name}_raw_score"] = df["raw_dssm_score"].astype("float32")
    df[f"{source_name}_pop_prior"] = df["pop_prior"].astype("float32")
    df[f"{source_name}_cat_l0_pref"] = df["cat_l0_pref"].astype("float32")
    df[f"{source_name}_catalog_pref"] = df["catalog_pref"].astype("float32")
    df[f"in_{source_name}"] = np.int8(1)
    df[f"f_num__candidate__in_{source_name}"] = np.float32(1.0)
    df[f"f_num__candidate__{source_name}_score"] = df["score"].astype("float32")
    df[f"f_num__candidate__{source_name}_rank"] = df["rank"].astype("float32")
    df[f"f_num__candidate__{source_name}_rr"] = (1.0 / df["rank"].astype("float32")).astype("float32")
    df[f"f_num__candidate__{source_name}_raw_score"] = df["raw_dssm_score"].astype("float32")
    df[f"f_num__candidate__{source_name}_pop_prior"] = df["pop_prior"].astype("float32")
    df[f"f_num__candidate__{source_name}_cat_l0_pref"] = df["cat_l0_pref"].astype("float32")
    df[f"f_num__candidate__{source_name}_catalog_pref"] = df["catalog_pref"].astype("float32")

    df["user_shard"] = (df["user_id"].astype("int64") % CFG.n_shards).astype("int16")
    written = 0
    for shard_id, part in df.groupby("user_shard", sort=True):
        shard_dir = out_dir / f"user_shard={int(shard_id)}"
        shard_dir.mkdir(parents=True, exist_ok=True)
        out_file = shard_dir / f"{source_name}_part_{part_id:05d}.parquet"
        pl.from_pandas(part.drop(columns=["user_shard"])).write_parquet(out_file, compression="zstd")
        written += len(part)
    return written


def generate_dl_candidates_for_split(model: nn.Module, ann: ANNIndex, split_name: str, item_id_by_idx: np.ndarray, item2idx: dict[int, int]) -> dict:
    out_dir = CANDIDATES_DIR / CFG.source_name / split_name
    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_dl_sources:
        print(f"{CFG.source_name}/{split_name}: source exists, reused -> {rel(out_dir)}")
        return {"split": split_name, "status": "reused", "rows": None, "out_dir": rel(out_dir)}

    if out_dir.exists() and (CFG.rebuild_dl_sources or CFG.rebuild_missing_dl_sources):
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

    target_users, user_seq, seen_by_user = build_sequences_for_split_with_checkpoint_vocab(split_name, item2idx)
    pop_prior = load_item_pop_prior_for_checkpoint_vocab(split_name, item_id_by_idx)
    catalog_arr = _safe_model_buffer_numpy(model, "catalog_idx_by_item_idx", len(item_id_by_idx))
    cat_l0_arr = _safe_model_buffer_numpy(model, "cat_l0_idx_by_item_idx", len(item_id_by_idx))

    rows_buffer: list[tuple] = []
    part_id = 0
    total_rows = 0
    t0 = time.perf_counter()

    for start in trange(0, len(target_users), CFG.retrieval_user_batch_size, desc=f"Generate {CFG.source_name}/{split_name}"):
        users = target_users[start:start + CFG.retrieval_user_batch_size]
        hists = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        keep = []
        for j, user_id in enumerate(users):
            seq = user_seq.get(int(user_id))
            if seq is None or len(seq) == 0:
                continue
            hist = seq[-CFG.sequence_max_len:]
            hists[j, -len(hist):] = hist.astype(np.int64)
            keep.append(j)
        if not keep:
            continue

        keep = np.asarray(keep, dtype=np.int64)
        batch_users = users[keep].astype(np.int64)
        batch_hists = hists[keep]
        user_vecs = encode_user_batch(model, batch_hists, batch_size=CFG.retrieval_user_batch_size)
        top_rows, top_scores = ann.query(user_vecs, CFG.ann_oversample)

        for local_row, user_id in enumerate(batch_users):
            hist_idx = batch_hists[local_row]
            l0_pref_map = _history_pref_maps(hist_idx, cat_l0_arr)
            catalog_pref_map = _history_pref_maps(hist_idx, catalog_arr)
            seen = seen_by_user.get(int(user_id), set()) if CFG.filter_seen_context_items else set()
            cand = []
            for ann_rank, (ridx, raw_score) in enumerate(zip(top_rows[local_row], top_scores[local_row]), start=1):
                ridx = int(ridx)
                item_idx = ridx + 1
                if item_idx <= 0 or item_idx >= len(item_id_by_idx):
                    continue
                item_id = int(item_id_by_idx[item_idx])
                if CFG.filter_seen_context_items and item_id in seen:
                    continue
                cat_l0_pref = float(l0_pref_map.get(int(cat_l0_arr[item_idx]), 0.0))
                catalog_pref = float(catalog_pref_map.get(int(catalog_arr[item_idx]), 0.0))
                pprior = float(pop_prior[item_idx])
                calibrated = (
                    float(raw_score)
                    + CFG.dssm_candidate_pop_alpha * pprior
                    + CFG.dssm_candidate_cat_l0_alpha * cat_l0_pref
                    + CFG.dssm_candidate_catalog_alpha * catalog_pref
                    - CFG.dssm_candidate_rank_alpha * math.log1p(float(ann_rank))
                )
                cand.append((calibrated, item_id, float(raw_score), pprior, cat_l0_pref, catalog_pref))
            cand.sort(key=lambda x: (-x[0], x[1]))
            for rank, (score, item_id, raw_score, pprior, cat_l0_pref, catalog_pref) in enumerate(cand[:CFG.dl_candidates_per_user], start=1):
                rows_buffer.append((int(user_id), int(item_id), float(score), int(rank), float(raw_score), float(pprior), float(cat_l0_pref), float(catalog_pref)))

        if len(rows_buffer) >= CFG.retrieval_flush_rows:
            total_rows += write_dl_buffer_to_shards(rows_buffer, out_dir, CFG.source_name, part_id)
            rows_buffer.clear()
            part_id += 1
            gc.collect()

    if rows_buffer:
        total_rows += write_dl_buffer_to_shards(rows_buffer, out_dir, CFG.source_name, part_id)
        rows_buffer.clear()

    meta = {
        "source_name": CFG.source_name,
        "split_name": split_name,
        "mode": "inference_only_from_checkpoint",
        "checkpoint_path": rel(get_checkpoint_path()),
        "checkpoint_split": CFG.checkpoint_split,
        "rows": int(total_rows),
        "users_with_history": int(len(target_users)),
        "ann_backend": ann.backend,
        "candidate_inference_time_sec": float(time.perf_counter() - t0),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()},
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"{CFG.source_name}/{split_name}: rows={total_rows:,} -> {rel(out_dir)}")
    return {"split": split_name, "status": "built", "rows": int(total_rows), "out_dir": rel(out_dir)}


In [6]:

# -----------------------------
# Safe merge with base-user filtering
# -----------------------------

def normalize_base_lf(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    if "candidate_rank" not in cols:
        if "rank" in cols:
            lf = lf.with_columns(pl.col("rank").cast(pl.Int32).alias("candidate_rank"))
        else:
            raise ValueError("Base candidate shard has neither candidate_rank nor rank")
    return lf


def normalize_dl_lf(lf: pl.LazyFrame, source: str) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs = []
    if "rank" not in cols and "candidate_rank" in cols:
        exprs.append(pl.col("candidate_rank").cast(pl.Int32).alias("rank"))
    if "score" not in cols and f"{source}_score" in cols:
        exprs.append(pl.col(f"{source}_score").cast(pl.Float32).alias("score"))
    if "source" not in cols:
        exprs.append(pl.lit(source).alias("source"))
    if f"{source}_score" not in cols and "score" in cols:
        exprs.append(pl.col("score").cast(pl.Float32).alias(f"{source}_score"))
    if f"{source}_rank" not in cols and "rank" in cols:
        exprs.append(pl.col("rank").cast(pl.Int32).alias(f"{source}_rank"))
    if f"in_{source}" not in cols:
        exprs.append(pl.lit(1, dtype=pl.Int8).alias(f"in_{source}"))
    if f"f_num__candidate__in_{source}" not in cols:
        exprs.append(pl.lit(1.0, dtype=pl.Float32).alias(f"f_num__candidate__in_{source}"))
    if f"f_num__candidate__{source}_score" not in cols:
        score_expr = pl.col(f"{source}_score") if f"{source}_score" in cols else pl.col("score")
        exprs.append(score_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_score"))
    if f"f_num__candidate__{source}_rank" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append(rank_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_rank"))
    if f"f_num__candidate__{source}_rr" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append((1.0 / rank_expr.cast(pl.Float32).clip(lower_bound=1.0)).cast(pl.Float32).alias(f"f_num__candidate__{source}_rr"))
    return lf.with_columns(exprs) if exprs else lf


def aggregate_merged_frame(df: pl.DataFrame, source: str) -> pl.DataFrame:
    agg_exprs = []
    for c in df.columns:
        if c in {"user_id", "item_id"}:
            continue
        if c in {"_base_present", "_dl_present"} or c.startswith("in_") or c.startswith("f_num__candidate__in_"):
            agg_exprs.append(pl.col(c).fill_null(0).max().alias(c))
        elif c in {"_base_rank", "_dl_rank", "candidate_rank", "rank", "min_source_rank"} or c.endswith("_rank"):
            agg_exprs.append(pl.col(c).min().alias(c))
        elif c in {"score", "max_source_score"} or c.endswith("_score") or c.endswith("_rr") or c.endswith("_prior") or c.endswith("_pref"):
            agg_exprs.append(pl.col(c).max().alias(c))
        elif c == "source":
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))
        else:
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))

    out = df.group_by(["user_id", "item_id"]).agg(agg_exprs)
    flag_cols = [c for c in out.columns if c.startswith("in_")]
    score_cols = [c for c in out.columns if c.endswith("_score") and not c.startswith("f_num__")]
    rank_cols = [c for c in out.columns if c.endswith("_rank") and not c.startswith("f_num__")]
    exprs = []
    if flag_cols:
        exprs.append(pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in flag_cols]).cast(pl.Int16).alias("sources_count"))
    if score_cols:
        exprs.append(pl.max_horizontal(*[pl.col(c).fill_null(-1e9).cast(pl.Float32) for c in score_cols]).cast(pl.Float32).alias("max_source_score"))
    if rank_cols:
        exprs.append(pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000).cast(pl.Int32) for c in rank_cols]).cast(pl.Int32).alias("min_source_rank"))
    return out.with_columns(exprs) if exprs else out


def safe_merge_split(split: str) -> dict:
    base_dir = resolve_base_candidate_dir(split)
    dl_dir = resolve_dl_candidate_dir(split)
    if base_dir is None:
        return {"split": split, "status": "skipped_no_base", "rows": 0, "users": 0}
    if dl_dir is None:
        return {"split": split, "status": "skipped_no_dl", "base_dir": rel(base_dir), "rows": 0, "users": 0}

    out_dir = CANDIDATES_DIR / CFG.output_merged_name / split
    if out_dir.exists() and CFG.rebuild_merged_with_dl:
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

    source = CFG.source_name
    rows_total = 0
    users_total = set()

    for shard_id in trange(CFG.n_shards, desc=f"Safe merge {split}"):
        frames = []
        base_lf = scan_candidate_shard_or_none(base_dir, shard_id)
        if base_lf is None:
            continue
        base_lf = normalize_base_lf(base_lf)
        base = collect_streaming(
            base_lf
            .filter(pl.col("candidate_rank").cast(pl.Int32) <= int(CFG.base_keep_per_user))
            .with_columns([
                pl.lit(1, dtype=pl.Int8).alias("_base_present"),
                pl.col("candidate_rank").fill_null(1_000_000).cast(pl.Int32).alias("_base_rank"),
                pl.lit(0, dtype=pl.Int8).alias("_dl_present"),
                pl.lit(1_000_000, dtype=pl.Int32).alias("_dl_rank"),
            ])
        )
        if base.is_empty():
            continue
        frames.append(base)
        base_users = base.select(pl.col("user_id").cast(pl.Int64).unique()).lazy()

        dl_lf = scan_candidate_shard_or_none(dl_dir, shard_id)
        if dl_lf is not None:
            dl_lf = normalize_dl_lf(dl_lf, source)
            rank_col = "rank" if "rank" in lazy_columns(dl_lf) else "candidate_rank"
            # Critical fix: keep only users from base split/shard. This prevents 880k-user val explosions.
            dl = collect_streaming(
                dl_lf
                .join(base_users, on="user_id", how="inner")
                .filter(pl.col(rank_col).cast(pl.Int32) <= int(CFG.dl_keep_per_user))
                .with_columns([
                    pl.lit(0, dtype=pl.Int8).alias("_base_present"),
                    pl.lit(1_000_000, dtype=pl.Int32).alias("_base_rank"),
                    pl.lit(1, dtype=pl.Int8).alias("_dl_present"),
                    pl.col(rank_col).fill_null(1_000_000).cast(pl.Int32).alias("_dl_rank"),
                ])
            )
            if not dl.is_empty():
                frames.append(dl)

        all_cols = sorted(set().union(*(set(f.columns) for f in frames)))
        aligned = []
        for f in frames:
            missing = [c for c in all_cols if c not in f.columns]
            if missing:
                f = f.with_columns([pl.lit(None).alias(c) for c in missing])
            aligned.append(f.select(all_cols))

        merged = aggregate_merged_frame(pl.concat(aligned, how="vertical_relaxed"), source)
        score_for_sort = "max_source_score" if "max_source_score" in merged.columns else (f"{source}_score" if f"{source}_score" in merged.columns else "score")
        merged = (
            merged
            .with_columns([
                pl.when(pl.col("_base_present").fill_null(0) > 0).then(0).otherwise(1).cast(pl.Int8).alias("_source_priority"),
                pl.col(score_for_sort).fill_null(-1e9).cast(pl.Float32).alias("_tmp_score"),
            ])
            .sort(
                ["user_id", "_source_priority", "_base_rank", "_dl_rank", "_tmp_score", "item_id"],
                descending=[False, False, False, False, True, False],
            )
            .with_columns(pl.arange(1, pl.len() + 1).over("user_id").cast(pl.Int32).alias("candidate_rank"))
            .filter(pl.col("candidate_rank") <= int(CFG.total_keep_per_user))
            .drop(["_source_priority", "_tmp_score"], strict=False)
        )
        shard_out = out_dir / f"user_shard={shard_id}"
        ensure_dir(shard_out)
        merged.write_parquet(shard_out / "part.parquet", compression="zstd")
        rows_total += merged.height
        users_total.update(int(u) for u in merged.select("user_id").unique().to_series().to_list())
        del frames, aligned, merged, base
        gc.collect()

    meta = {
        "split": split,
        "status": "merged",
        "base_dir": rel(base_dir),
        "dl_dir": rel(dl_dir),
        "out_dir": rel(out_dir),
        "rows": int(rows_total),
        "users": int(len(users_total)),
        "base_keep_per_user": int(CFG.base_keep_per_user),
        "dl_keep_per_user": int(CFG.dl_keep_per_user),
        "total_keep_per_user": int(CFG.total_keep_per_user),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return meta


In [7]:

# -----------------------------
# Validation metrics and reports
# -----------------------------

def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .filter(pl.col("cnt_orders").fill_null(0) > 0)
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_id"])
    )
    result: dict[int, list[int]] = defaultdict(list)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].append(int(i))
    return dict(result)


def _unique_int_list(values) -> list[int]:
    if values is None:
        return []
    return list(dict.fromkeys(int(x) for x in values))


def hit_rate_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(any(int(x) in gt for x in pred_items[:k])) if gt else np.nan


def recall_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(len(gt & set(int(x) for x in pred_items[:k])) / len(gt)) if gt else np.nan


def ndcg_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return float(dcg / idcg) if idcg else 0.0


def ap_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    hits = 0
    score = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            hits += 1
            score += hits / rank
    return float(score / len(gt))


def rr_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            return float(1.0 / rank)
    return 0.0


def evaluate_candidate_dir(candidates_dir: Path, ground_truth: dict[int, list[int]], *, ks: Iterable[int], rank_col: str, label: str) -> pd.DataFrame:
    if not candidate_dir_looks_complete(candidates_dir):
        return pd.DataFrame()
    ks = tuple(int(k) for k in ks)
    sums = {k: defaultdict(float) for k in ks}
    candidate_counts = []
    users_eval = 0
    unique_items: set[int] = set()
    for shard_id in trange(CFG.n_shards, desc=f"Evaluate {label}"):
        lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("rank" if "rank" in cols else "candidate_rank")
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(actual_rank_col).cast(pl.Int32).alias("eval_rank"),
            ).sort(["user_id", "eval_rank", "item_id"])
        )
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for user_id, group in pdf.groupby("user_id", sort=False):
            user_id = int(user_id)
            gt = ground_truth.get(user_id)
            if not gt:
                continue
            pred = group["item_id"].astype(np.int64).tolist()
            users_eval += 1
            candidate_counts.append(len(pred))
            unique_items.update(pred[:max(ks)])
            for k in ks:
                sums[k]["hit_rate"] += hit_rate_at_k(gt, pred, k)
                sums[k]["recall"] += recall_at_k(gt, pred, k)
                sums[k]["ndcg"] += ndcg_at_k(gt, pred, k)
                sums[k]["map"] += ap_at_k(gt, pred, k)
                sums[k]["mrr"] += rr_at_k(gt, pred, k)
        del pdf, part
        gc.collect()
    rows = []
    denom = max(users_eval, 1)
    for k in ks:
        rows.append({
            "model": label,
            "k": k,
            "users_eval": int(users_eval),
            "hit_rate": sums[k]["hit_rate"] / denom,
            "recall": sums[k]["recall"] / denom,
            "ndcg": sums[k]["ndcg"] / denom,
            "map": sums[k]["map"] / denom,
            "mrr": sums[k]["mrr"] / denom,
            "avg_candidates_per_user": float(np.mean(candidate_counts)) if candidate_counts else 0.0,
            "p50_candidates_per_user": float(np.percentile(candidate_counts, 50)) if candidate_counts else 0.0,
            "p95_candidates_per_user": float(np.percentile(candidate_counts, 95)) if candidate_counts else 0.0,
            "unique_items": int(len(unique_items)),
        })
    return pd.DataFrame(rows)


def build_positive_pairs_df(gt: dict[int, list[int]]) -> pl.DataFrame:
    rows = [(int(u), int(i)) for u, items in gt.items() for i in items]
    return pl.DataFrame(rows, schema={"user_id": pl.Int64, "item_id": pl.Int64}, orient="row").unique() if rows else pl.DataFrame({"user_id": [], "item_id": []}, schema={"user_id": pl.Int64, "item_id": pl.Int64})


def collect_found_positive_pairs_at_k(path: Path, gt_pairs: pl.DataFrame, k: int, *, rank_col: str) -> set[tuple[int, int]]:
    found: set[tuple[int, int]] = set()
    if not candidate_dir_looks_complete(path) or gt_pairs.is_empty():
        return found
    gt_lf = gt_pairs.lazy()
    for shard_id in trange(CFG.n_shards, desc=f"positive pairs {path.parent.name}/{path.name}@{k}"):
        lf = scan_candidate_shard_or_none(path, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("rank" if "rank" in cols else "candidate_rank")
        hits = collect_streaming(
            lf.select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), pl.col(actual_rank_col).cast(pl.Int32).alias("r"))
            .filter(pl.col("r") <= int(k))
            .join(gt_lf, on=["user_id", "item_id"], how="inner")
            .select("user_id", "item_id")
            .unique()
        )
        if not hits.is_empty():
            found.update((int(u), int(i)) for u, i in zip(hits["user_id"].to_numpy(), hits["item_id"].to_numpy()))
    return found


In [8]:

# -----------------------------
# Run: build missing DL sources, safe merge, metrics
# -----------------------------

# 1) Resolve current state.
state_rows = []
for split in CFG.splits_to_merge:
    base = resolve_base_candidate_dir(split)
    dl = resolve_dl_candidate_dir(split)
    out = CANDIDATES_DIR / CFG.output_merged_name / split
    state_rows.append({
        "split": split,
        "base_dir": rel(base) if base else None,
        "base_exists": bool(base),
        "dl_dir": rel(dl) if dl else None,
        "dl_exists": bool(dl),
        "merged_with_dl_dir": rel(out),
        "merged_with_dl_exists": candidate_dir_looks_complete(out),
    })
state = pd.DataFrame(state_rows)
display(state)
state.to_parquet(REPORTS_DIR / "dl_candidates_inference_initial_state.parquet", index=False)

# 2) Load model/checkpoint once.
model, item_id_by_idx, item2idx = load_dssm_checkpoint()
ann = load_or_build_ann_index(model, item_id_by_idx)

# 3) Generate missing DL candidates for configured splits.
build_rows = []
for split in CFG.splits_to_build_dl:
    out_dir = CANDIDATES_DIR / CFG.source_name / split
    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_dl_sources:
        print(f"{split}: DL source already exists, skip generation -> {rel(out_dir)}")
        build_rows.append({"split": split, "status": "reused", "out_dir": rel(out_dir)})
        continue
    build_rows.append(generate_dl_candidates_for_split(model, ann, split, item_id_by_idx, item2idx))

build_report = pd.DataFrame(build_rows)
display(build_report)
build_report.to_parquet(REPORTS_DIR / "dl_candidates_inference_build_report.parquet", index=False)
build_report.to_csv(REPORTS_DIR / "dl_candidates_inference_build_report.csv", index=False)
print("saved:", rel(REPORTS_DIR / "dl_candidates_inference_build_report.parquet"))

# 4) Safe merge all splits. Existing val DL source is filtered to base users during merge.
merge_rows = []
for split in CFG.splits_to_merge:
    meta = safe_merge_split(split)
    merge_rows.append(meta)
merge_report = pd.DataFrame(merge_rows)
display(merge_report)
merge_report.to_parquet(REPORTS_DIR / "dl_candidates_merge_report.parquet", index=False)
merge_report.to_csv(REPORTS_DIR / "dl_candidates_merge_report.csv", index=False)
print("saved:", rel(REPORTS_DIR / "dl_candidates_merge_report.parquet"))

# 5) Validation metrics and reports.
if CFG.evaluate_val:
    gt_val = build_ground_truth(CFG.val_target_start, CFG.val_target_end)
    base_val = resolve_base_candidate_dir("val")
    dl_val = resolve_dl_candidate_dir("val")
    merged_val = CANDIDATES_DIR / CFG.output_merged_name / "val"
    frames = []
    if base_val is not None:
        frames.append(evaluate_candidate_dir(base_val, gt_val, ks=CFG.metric_ks, rank_col="candidate_rank", label="base_non_dl"))
    if dl_val is not None:
        frames.append(evaluate_candidate_dir(dl_val, gt_val, ks=CFG.metric_ks, rank_col="rank", label=CFG.source_name))
    if candidate_dir_looks_complete(merged_val):
        frames.append(evaluate_candidate_dir(merged_val, gt_val, ks=CFG.metric_ks, rank_col="candidate_rank", label=CFG.output_merged_name))
    val_metrics = pd.concat([f for f in frames if f is not None and not f.empty], ignore_index=True) if frames else pd.DataFrame()
    display(val_metrics)
    val_metrics.to_parquet(REPORTS_DIR / "dl_candidates_val_metrics.parquet", index=False)
    val_metrics.to_csv(REPORTS_DIR / "dl_candidates_val_metrics.csv", index=False)
    print("saved:", rel(REPORTS_DIR / "dl_candidates_val_metrics.parquet"))

    if CFG.run_unique_positive_report and base_val is not None and dl_val is not None:
        gt_pairs = build_positive_pairs_df(gt_val)
        total_pos = gt_pairs.height
        base_pos = collect_found_positive_pairs_at_k(base_val, gt_pairs, CFG.base_keep_per_user, rank_col="candidate_rank")
        dl_pos = collect_found_positive_pairs_at_k(dl_val, gt_pairs, CFG.dl_keep_per_user, rank_col="rank")
        merged_pos = collect_found_positive_pairs_at_k(merged_val, gt_pairs, CFG.total_keep_per_user, rank_col="candidate_rank") if candidate_dir_looks_complete(merged_val) else set()
        unique_report = pd.DataFrame([{
            "base_k": int(CFG.base_keep_per_user),
            "dl_k": int(CFG.dl_keep_per_user),
            "merged_k": int(CFG.total_keep_per_user),
            "target_positive_pairs": int(total_pos),
            "base_found_pairs": int(len(base_pos)),
            "dssm_found_pairs": int(len(dl_pos)),
            "dssm_unique_vs_base_pairs": int(len(dl_pos - base_pos)),
            "overlap_positive_pairs": int(len(base_pos & dl_pos)),
            "union_found_pairs": int(len(base_pos | dl_pos)),
            "merged_found_pairs": int(len(merged_pos)),
            "base_pair_recall": float(len(base_pos) / total_pos) if total_pos else np.nan,
            "dssm_pair_recall": float(len(dl_pos) / total_pos) if total_pos else np.nan,
            "incremental_pair_recall": float(len(dl_pos - base_pos) / total_pos) if total_pos else np.nan,
            "union_pair_recall": float(len(base_pos | dl_pos) / total_pos) if total_pos else np.nan,
            "merged_pair_recall": float(len(merged_pos) / total_pos) if total_pos else np.nan,
        }])
        display(unique_report)
        unique_report.to_parquet(REPORTS_DIR / "dl_candidates_unique_positives.parquet", index=False)
        unique_report.to_csv(REPORTS_DIR / "dl_candidates_unique_positives.csv", index=False)
        print("saved:", rel(REPORTS_DIR / "dl_candidates_unique_positives.parquet"))

        if CFG.run_quota_ablation:
            quota_rows = []
            for quota in CFG.quota_values:
                dl_q = min(int(quota), int(CFG.dl_keep_per_user))
                dl_pos_q = collect_found_positive_pairs_at_k(dl_val, gt_pairs, dl_q, rank_col="rank")
                union_q = base_pos | dl_pos_q
                quota_rows.append({
                    "base_k": int(CFG.base_keep_per_user),
                    "dssm_quota": int(dl_q),
                    "total_candidate_cap": int(CFG.base_keep_per_user + dl_q),
                    "dssm_found_pairs": int(len(dl_pos_q)),
                    "dssm_unique_vs_base_pairs": int(len(dl_pos_q - base_pos)),
                    "union_found_pairs": int(len(union_q)),
                    "incremental_pair_recall": float(len(dl_pos_q - base_pos) / total_pos) if total_pos else np.nan,
                    "union_pair_recall": float(len(union_q) / total_pos) if total_pos else np.nan,
                })
            quota_ablation = pd.DataFrame(quota_rows)
            display(quota_ablation)
            quota_ablation.to_parquet(REPORTS_DIR / "dl_candidates_quota_ablation.parquet", index=False)
            quota_ablation.to_csv(REPORTS_DIR / "dl_candidates_quota_ablation.csv", index=False)
            print("saved:", rel(REPORTS_DIR / "dl_candidates_quota_ablation.parquet"))

# 6) Manifest.
manifest = {
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "inference_only_from_existing_dssm_checkpoint",
    "checkpoint_path": rel(get_checkpoint_path()),
    "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()},
    "outputs": {
        split: rel(CANDIDATES_DIR / CFG.output_merged_name / split)
        for split in CFG.splits_to_merge
        if candidate_dir_looks_complete(CANDIDATES_DIR / CFG.output_merged_name / split)
    },
    "dl_sources": {
        split: rel(CANDIDATES_DIR / CFG.source_name / split)
        for split in CFG.splits_to_merge
        if candidate_dir_looks_complete(CANDIDATES_DIR / CFG.source_name / split)
    },
    "reports": {
        "build_report": rel(REPORTS_DIR / "dl_candidates_inference_build_report.parquet"),
        "merge_report": rel(REPORTS_DIR / "dl_candidates_merge_report.parquet"),
        "val_metrics": rel(REPORTS_DIR / "dl_candidates_val_metrics.parquet"),
        "unique_positives": rel(REPORTS_DIR / "dl_candidates_unique_positives.parquet"),
        "quota_ablation": rel(REPORTS_DIR / "dl_candidates_quota_ablation.parquet"),
    },
}
manifest_path = CANDIDATES_DIR / "manifest_dl_candidates_with_dssm_final.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(manifest_path))
print(json.dumps(manifest, ensure_ascii=False, indent=2))


,split,base_dir,base_exists,dl_dir,dl_exists,merged_with_dl_dir,merged_with_dl_exists
0,ranker_train,data\datamart\candidates\merged\ranker_train,True,NaN,False,data\datamart\candidates\merged_with_dl\ranker...,False
1,val,data\datamart\candidates\merged\val,True,data\datamart\candidates\dl_dssm_final\val,True,data\datamart\candidates\merged_with_dl\val,True
2,test,data\datamart\candidates\merged\test,True,NaN,False,data\datamart\candidates\merged_with_dl\test,False


Load checkpoint: models\dl_candidate_generators_dssm_final\dl_dssm_final\val\model.pt


C:\Users\denis\AppData\Local\Temp\ipykernel_54900\2726446332.py:70: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  self.register_buffer("fclip_table", torch.from_numpy(clip_arr), persistent=False)


Checkpoint loaded: items=1,200,000, source=dl_dssm_final
HNSW loaded: models\dl_candidate_generators_dssm_final\dl_dssm_final\val\dl_dssm_final_val_hnsw.bin


Read base users ranker_train:   0%|          | 0/128 [00:00<?, ?it/s]

ranker_train: base users saved -> data\processed\dl_candidate_generators_dssm_final\inference_from_checkpoint\base_users\base_users_ranker_train.npy, users=376,016
ranker_train: build context sequences with checkpoint item vocab, context=2025-04-18..2025-06-03, base_users=376,016
ranker_train: checkpoint-vocab sequences saved -> data\processed\dl_candidate_generators_dssm_final\inference_from_checkpoint\sequences\ranker_train, users=373,580


... (25 outputs left out)

positive pairs dl_dssm_final/val@200:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs dl_dssm_final/val@300:   0%|          | 0/128 [00:00<?, ?it/s]

,base_k,dssm_quota,total_candidate_cap,dssm_found_pairs,dssm_unique_vs_base_pairs,union_found_pairs,incremental_pair_recall,union_pair_recall
0,700,50,750,4937,2865,143663,0.001817,0.091116
1,700,100,800,8316,4961,145759,0.003146,0.092445
2,700,200,900,13656,8353,149151,0.005298,0.094596
3,700,300,1000,18303,11457,152255,0.007266,0.096565


saved: artifacts\reports\dl_candidates_quota_ablation.parquet
saved: data\datamart\candidates\manifest_dl_candidates_with_dssm_final.json
{
  "built_at_utc": "2026-05-10T09:18:41.073157+00:00",
  "mode": "inference_only_from_existing_dssm_checkpoint",
  "checkpoint_path": "models\\dl_candidate_generators_dssm_final\\dl_dssm_final\\val\\model.pt",
  "config": {
    "ranker_train_context_start": "2025-04-18",
    "ranker_train_context_end": "2025-06-03",
    "ranker_train_target_start": "2025-06-04",
    "ranker_train_target_end": "2025-06-17",
    "val_context_start": "2025-04-18",
    "val_context_end": "2025-06-17",
    "val_target_start": "2025-06-18",
    "val_target_end": "2025-07-01",
    "full_context_start": "2025-05-02",
    "full_context_end": "2025-07-01",
    "test_target_start": "2025-07-02",
    "test_target_end": "2025-07-15",
    "n_shards": 128,
    "source_name": "dl_dssm_final",
    "output_merged_name": "merged_with_dl",
    "checkpoint_split": "val",
    "checkpoint